In [6]:
import tensorflow as tf
import numpy as np
import pandas as pd
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error

In [7]:
data_dir = "data/"
train_features = pd.read_csv(f'{data_dir}train_processed.csv')
train_consumption = pd.read_csv(f'{data_dir}train_hh_gt.csv')
train_features['cons_ppp17'] = train_consumption['cons_ppp17']
df_clean = train_features.dropna()
x=df_clean.drop(columns=['cons_ppp17','weight'])
y=df_clean[['cons_ppp17']]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)
x_train.shape , x_test.shape

((72888, 71), (31238, 71))

In [8]:
scaler = StandardScaler()
target_scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

y_train_scaled = target_scaler.fit_transform(y_train)
y_test_scaled = target_scaler.transform(y_test)

In [15]:
def build_model(hp):
    model= keras.Sequential()
    model.add(keras.Input(shape=(71,)))
    model.add(layers.Dense(units=hp.Choice('units_1',values=[64,128]),activation='relu'))
    if hp.Boolean('dropout_1'):
        model.add(layers.Dropout(0.2))
    model.add(layers.Dense(units=hp.Choice('units_2',values=[32,64]),activation='relu'))
    if hp.Boolean('dropout_2'):
        model.add(layers.Dropout(0.2))
    model.add(layers.Dense(units=32, activation='relu'))
    model.add(layers.Dense(1))
    lr =hp.Choice('learning_rate', values=[0.001, 0.0001])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss='mse',metrics=['mae'])
    return model

In [16]:
tuner= kt.RandomSearch(build_model, objective='val_loss',max_trials=20,directory='tuning_dir',project_name='ml_comp')

In [17]:
stop_early=tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

In [18]:
tuner.search(x_train_scaled,y_train_scaled,epochs=50,validation_split=0.2,callbacks=[stop_early])

Trial 20 Complete [00h 02m 55s]
val_loss: 0.40783047676086426

Best val_loss So Far: 0.401792973279953
Total elapsed time: 00h 42m 47s


In [19]:
best_model=tuner.get_best_models(num_models=1)[0]

C:\Users\giorg\anaconda3\envs\tf\lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [26]:
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 128)                 │           9,216 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,585 (76.50 KB)

 Trainable params: 19,585 (76.50 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
final_predictions=best_model.predict(x_test_scaled)

977/977 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


In [21]:
y_pred_original =target_scaler.inverse_transform(final_predictions.reshape(-1, 1))

In [23]:
mse_list = []
mae_list = []
r2_list = []
models = []

In [24]:
y_pred_fixed = np.maximum(y_pred_original, 0)
mse = mean_squared_error(y_test, y_pred_fixed)
mae =mean_absolute_error(y_test, y_pred_fixed)
r2=r2_score(y_test, y_pred_fixed)

mse_list.append(mse)
mae_list.append(mae)
r2_list.append(r2)
models.append('DL')

print(f'MSE: {mse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}')

MSE: 42.02, MAE: 3.61, R2: 0.59


In [27]:
best_model.save('tuned_dl_model.h5')
print("Model saved!")

Model saved!
